### Cell 1 — Markdown
# Notebook 05 — Final Data Preparation

**Objective:**  
Prepare clean, analysis-ready datasets for Tableau dashboard and final reporting.

Outputs:
- Customer-level dataset (segmentation & KPIs)
- Order-level dataset (time-series & trends)

In [1]:
import pandas as pd

print("Libraries loaded")

Libraries loaded


In [2]:
df = pd.read_csv("../data/processed/olist_master_cleaned.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (96469, 32)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_weight_g,seller_city,seller_state,delivery_time_days,delivery_delay_days,is_late_delivery,total_order_value,customer_repeat_flag,order_month,order_year
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,500.0,maua,SP,8.0,-8.0,False,38.71,True,2017-10,2017
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,400.0,belo horizonte,SP,13.0,-6.0,False,141.46,False,2018-07,2018
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,420.0,guariba,SP,9.0,-18.0,False,179.12,False,2018-08,2018
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,450.0,belo horizonte,MG,13.0,-13.0,False,72.20,False,2017-11,2017
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,250.0,mogi das cruzes,SP,2.0,-10.0,False,28.62,False,2018-02,2018


CUSTOMER-LEVEL DATASET

In [3]:
customer_df = df.groupby('customer_unique_id').agg({
    'order_id': 'nunique',
    'total_order_value': 'sum',
    'review_score': 'mean',
    'delivery_delay_days': 'mean',
    'is_late_delivery': 'mean'
}).reset_index()

customer_df.columns = [
    'customer_id',
    'total_orders',
    'total_spent',
    'avg_review_score',
    'avg_delay',
    'late_delivery_rate'
]

print("Customer dataset created:", customer_df.shape)

Customer dataset created: (93349, 6)


In [4]:
customer_df['value_segment'] = pd.qcut(
    customer_df['total_spent'],
    q=3,
    labels=['Low Value', 'Mid Value', 'High Value']
)

print("\nSegment distribution:")
print(customer_df['value_segment'].value_counts())


Segment distribution:
value_segment
Low Value     31119
Mid Value     31117
High Value    31113
Name: count, dtype: int64


In [5]:
geo_df = df[['customer_unique_id', 'customer_city', 'customer_state']].drop_duplicates()

customer_df = customer_df.merge(
    geo_df,
    left_on='customer_id',
    right_on='customer_unique_id',
    how='left'
).drop(columns=['customer_unique_id'])

In [6]:
payment_df = df.groupby('customer_unique_id').agg({
    'payment_type': lambda x: x.mode()[0],
    'payment_installments': 'mean'
}).reset_index()

customer_df = customer_df.merge(
    payment_df,
    left_on='customer_id',
    right_on='customer_unique_id',
    how='left'
).drop(columns=['customer_unique_id'])

In [7]:
customer_df = customer_df.drop_duplicates()

customer_df['avg_review_score'] = customer_df['avg_review_score'].fillna(
    customer_df['avg_review_score'].median()
)

customer_df['avg_delay'] = customer_df['avg_delay'].fillna(0)
customer_df['late_delivery_rate'] = customer_df['late_delivery_rate'].fillna(0)

print("\nNulls after cleaning:")
print(customer_df.isnull().sum())

print("\nFinal shape:", customer_df.shape)


Nulls after cleaning:
customer_id             0
total_orders            0
total_spent             0
avg_review_score        0
avg_delay               0
late_delivery_rate      0
value_segment           0
customer_city           0
customer_state          0
payment_type            0
payment_installments    0
dtype: int64

Final shape: (93462, 11)


In [8]:
customer_df.to_csv("../data/processed/tableau_customer_dataset.csv", index=False)

print("Customer dataset exported")

Customer dataset exported


ORDER-LEVEL DATASET (FOR TABLEAU)

In [9]:
tableau_orders = df[[
    'order_id',
    'customer_unique_id',
    'order_purchase_timestamp',
    'customer_state',
    'customer_city',
    'product_category_name_english',
    'total_order_value',
    'payment_type',
    'payment_installments',
    'review_score',
    'delivery_time_days',
    'delivery_delay_days',
    'is_late_delivery'
]].copy()

print("Order dataset created:", tableau_orders.shape)

Order dataset created: (96469, 13)


In [10]:
tableau_orders.to_csv("../data/processed/tableau_orders_dataset.csv", index=False)

print("Order dataset exported")

Order dataset exported


FINAL VALIDATION

In [11]:
print("\nCustomer Dataset Preview:")
print(customer_df.head())

print("\nOrder Dataset Preview:")
print(tableau_orders.head())


Customer Dataset Preview:
                        customer_id  total_orders  total_spent  \
0  0000366f3b9a7992bf8c76cfdf3221e2             1       141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f             1        27.19   
2  0000f46a3911fa3c0805444483337064             1        86.22   
3  0000f6ccb0745a6a4b88665a16c9f078             1        43.62   
4  0004aac84e0df4da2b147fca70cf8255             1       196.89   

   avg_review_score  avg_delay  late_delivery_rate value_segment  \
0               5.0       -5.0                 0.0     Mid Value   
1               4.0       -5.0                 0.0     Low Value   
2               3.0       -2.0                 0.0     Mid Value   
3               4.0      -12.0                 0.0     Low Value   
4               5.0       -8.0                 0.0    High Value   

  customer_city customer_state payment_type  payment_installments  
0       cajamar             SP  credit_card                   8.0  
1        osasco             SP 

### Cell 13 — Markdown Summary
## Final Export Summary

| File | Purpose |
|------|--------|
| tableau_customer_dataset.csv | Customer segmentation & KPIs |
| tableau_orders_dataset.csv | Time-series & delivery analysis |

**Next Step:** Build Tableau dashboard using both datasets.